# Task 4 — GPT Language Model from Scratch

## Introduction

This notebook builds and evaluates a small **GPT-style language model from scratch** using the Shakespeare dataset. It reuses the BPE tokenization and baseline language-model components from the earlier tasks, then compares them with a causal Transformer model. The text-processing utilities preserve punctuation by treating punctuation marks as separate units during normalization, so the models can learn punctuation patterns as part of the language.

### What this notebook does

1. **Sets up the experiment** by importing the required libraries, fixing random seeds, selecting the available device (CUDA, MPS, or CPU), and creating folders for saved runs.
2. **Loads the Shakespeare train, validation, and test splits** and records file information so experiments can be reproduced.
3. **Reuses the Task 1–3 utilities** for BPE tokenization, the n-gram baseline, and the neural bigram baseline.
4. **Tests several BPE vocabulary sizes** by changing the number of BPE merges. A 4-gram model is used to measure validation perplexity, and the three best merge settings are selected.
5. **Evaluates baseline models** using a traditional 4-gram language model and a simple neural bigram language model.
6. **Defines a GPT-style Transformer** with token and positional embeddings, masked multi-head self-attention, feed-forward layers, residual connections, and layer normalization.
7. **Trains GPT models** for the selected BPE settings, monitors validation loss, uses early stopping, and saves experiment results/checkpoints.
8. **Evaluates the final language model** using perplexity on held-out data and compares its result with the baseline models.
9. **Generates sample text** autoregressively from the prompt *“To be, or not to be”* using temperature and top-k sampling.

### Main evaluation measure

**Perplexity (PPL)** is used to compare the language models. A lower perplexity means the model assigns higher probability to the correct next tokens and is therefore less uncertain about the text. Validation data is used for model/tokenizer selection, while the test split is used for the final evaluation.


In [1]:
#@title Setup: imports, seeds, device
import os, sys, math, time, json, random, csv, pickle, shutil, hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, List, Tuple, Dict, Any
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Seed Python, NumPy, and PyTorch before constructing models or shuffling data.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 1337
set_seed(SEED)

# Use the fastest available compute backend, with CPU as the final fallback.
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cuda


In [2]:
#@title Setup: persistent ./runs and RunManager helpers
PERSISTENT = True  # set False for ephemeral runs
DRIVE_RUNS_SUBDIR = "colab_runs/gpt_from_scratch"  # in MyDrive

# Detect whether the notebook is running inside Google Colab.
def in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

# Mount Drive only when persistent storage is requested and Colab is available.
def mount_drive_if_needed():
    if PERSISTENT and in_colab():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print("Drive mounted.")
    else:
        print("Not using Drive persistence.")

# Point /content/runs at persistent Drive storage or at a local temporary directory.
def ensure_runs_link():
    runs_link = Path("/content/runs")
    if PERSISTENT and in_colab():
        target = Path("/content/drive/MyDrive") / DRIVE_RUNS_SUBDIR
    else:
        target = Path("/content/runs_ephemeral")
    target.mkdir(parents=True, exist_ok=True)

    if runs_link.exists() or runs_link.is_symlink():
        if runs_link.is_symlink():
            if runs_link.resolve() == target.resolve():
                print(f"./runs already -> {target}")
            else:
                runs_link.unlink(); runs_link.symlink_to(target, target_is_directory=True)
        else:
            # rotate existing folder/file
            ts = datetime.now().strftime("%Y%m%d-%H%M%S")
            runs_link.rename(runs_link.with_name(f"runs_rotated_{ts}"))
            runs_link.symlink_to(target, target_is_directory=True)
    else:
        runs_link.symlink_to(target, target_is_directory=True)
    return runs_link

mount_drive_if_needed()
RUNS = ensure_runs_link()
print("RUNS root:", RUNS.resolve())

# Keep each experiment's configuration, logs, plots, and checkpoints in one run folder.
class RunManager:
    def __init__(self, base_link: Path = Path("/content/runs")):
        self.base = base_link
        self.run_dir: Optional[Path] = None

    # Start a timestamped run and save its configuration for reproducibility.
    def new(self, name_prefix: str, cfg: Dict[str, Any]) -> Path:
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        self.run_dir = self.base / f"{ts}-{name_prefix}"
        (self.run_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
        (self.run_dir / "plots").mkdir(exist_ok=True)
        (self.run_dir / "logs").mkdir(exist_ok=True)
        with open(self.run_dir / "config.json", "w") as f:
            json.dump(cfg, f, indent=2)
        print("Run dir:", self.run_dir)
        return self.run_dir

    # Append one metrics row to a CSV file, adding the header only once.
    def log_row(self, fname: str, row: Dict[str, Any], header: Optional[List[str]]=None):
        path = self.run_dir / "logs" / fname
        write_header = (not path.exists()) and header
        with open(path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header or row.keys())
            if write_header: w.writeheader()
            w.writerow(row)

    # Save and close a matplotlib figure so repeated experiments do not accumulate open figures.
    def save_plot(self, fig: plt.Figure, name: str):
        p = self.run_dir / "plots" / name
        fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig)
        return p

    # Store a model/training checkpoint using the training step in the filename.
    def save_ckpt(self, state: Dict[str, Any], step: int, tag="gpt"):
        fp = self.run_dir / "checkpoints" / f"{tag}_step{step:06d}.pt"
        torch.save(state, fp)
        return fp

RUN = RunManager(RUNS)


Mounted at /content/drive
Drive mounted.
RUNS root: /content/drive/MyDrive/colab_runs/gpt_from_scratch


In [3]:
#@title Data: load your Shakespeare splits (Drive or upload)
USE_DRIVE_DATA = True  # set False to upload
DRIVE_DATA_SUBDIR = "/content/drive/MyDrive/gptfromscratch/"  # in MyDrive

# Define the expected split filenames and an optional full-corpus fallback.
FN_TRAIN = "Shakespeare_clean_train.txt"
FN_VALID = "Shakespeare_clean_valid.txt"
FN_TEST  = "Shakespeare_clean_test.txt"
FN_FULL  = "Shakespeare_clean_full.txt"  # optional fallback

def ensure_dir(p: Path): p.mkdir(parents=True, exist_ok=True); return p
# Compute a short file hash so the exact dataset files can be identified in the output.
def sha1(p: Path):
    h=hashlib.sha1()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()[:10]

# Choose between Drive-backed data and files uploaded into the current Colab session.
if USE_DRIVE_DATA and in_colab():
    DATA_DIR = Path("/content/drive/MyDrive") / DRIVE_DATA_SUBDIR
    ensure_dir(DATA_DIR)
else:
    DATA_DIR = Path("/content/user_uploads")
    ensure_dir(DATA_DIR)
    # upload if empty
    if not any(DATA_DIR.iterdir()):
        from google.colab import files
        print("Please upload your Shakespeare files…")
        uploaded = files.upload()
        for name, content in uploaded.items():
            (DATA_DIR / name).write_bytes(content)

# Reuse existing splits when present; otherwise create deterministic train/valid/test splits.
def split_from_full_if_needed(train_ratio=0.98, valid_ratio=0.01, seed=1337):
    train_p = DATA_DIR / FN_TRAIN
    valid_p = DATA_DIR / FN_VALID
    test_p  = DATA_DIR / FN_TEST
    if train_p.exists() and valid_p.exists() and test_p.exists():
        return train_p, valid_p, test_p
    full = DATA_DIR / FN_FULL
    if not full.exists(): return train_p, valid_p, test_p
    print("Found only FULL — splitting deterministically…")
    lines = full.read_text(encoding="utf-8").splitlines(keepends=True)
    rng = random.Random(seed); idx = list(range(len(lines))); rng.shuffle(idx)
    n = len(lines); n_train = int(n*train_ratio); n_valid = int(n*valid_ratio); n_test = n-n_train-n_valid
    def write_subset(fn, idxs): (DATA_DIR/fn).write_text("".join(lines[i] for i in idxs), encoding="utf-8")
    write_subset(FN_TRAIN, idx[:n_train])
    write_subset(FN_VALID, idx[n_train:n_train+n_valid])
    write_subset(FN_TEST,  idx[n_train+n_valid:])
    print(f"Split sizes: train={n_train}, valid={n_valid}, test={n_test}")
    return train_p, valid_p, test_p

# Resolve the three split paths before checking that every required file exists.
TRAIN_PATH, VALID_PATH, TEST_PATH = split_from_full_if_needed()

for p in (TRAIN_PATH, VALID_PATH, TEST_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

# Load each split as raw text for tokenizer training and language-model evaluation.
train_text = TRAIN_PATH.read_text(encoding="utf-8")
valid_text = VALID_PATH.read_text(encoding="utf-8")
test_text  = TEST_PATH.read_text(encoding="utf-8")

def info(p, name):
    print(f"{name:<5} | {p.name:<30} | {p.stat().st_size:>9} bytes | sha1 {sha1(p)}")
info(TRAIN_PATH,"train"); info(VALID_PATH,"valid"); info(TEST_PATH,"test")
print("train sample:", repr(train_text[:120]))


train | Shakespeare_clean_train.txt    |    864424 bytes | sha1 a7c8fadc20
valid | Shakespeare_clean_valid.txt    |    104274 bytes | sha1 cc371fb31f
test  | Shakespeare_clean_test.txt     |    103975 bytes | sha1 23195995f9
train sample: 'The Tragedy of Antony and Cleopatra Dramatis Personae MARK ANTONY OCTAVIUS CAESAR M. AEMILIUS LEPIDUS triumvirs. SEXTUS '


In [4]:
'''
# === Cell 4 — Reuse Task 1–3 as in-notebook modules ===
# This cell writes: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py
# and imports them so the rest of the notebook just works.

from pathlib import Path
import importlib

# -------- utils_bpe.py (Task 1) --------
utils_bpe_src = r'''
import re
from collections import defaultdict
from typing import List, Tuple, Dict

# ---- Task 1 helpers ----

def normalize_text(
    text: str,
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
) -> str:
    if use_only_lowercase:
        text = text.lower()
    if use_only_alphabetic:
        text = re.sub(r"[^a-z\s]", "", text)
    if make_the_puncs_as_a_single_word:
        text = re.sub(r'([.,!?;:()\[\]{}"\'\-])', r" \1 ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text

def prepare_text_for_bpe(text: str, *_args, **_kwargs) -> Dict[str, int]:
    words = text.split(" ")
    vocab = defaultdict(int)
    for w in words:
        if w:
            vocab[" ".join(list(w)) + " _"] += 1
    return vocab

def get_stats(vocab: Dict[str, int]):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair: Tuple[str,str], v_in: Dict[str,int]) -> Dict[str,int]:
    v_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for w in v_in:
        v_out[p.sub("".join(pair), w)] = v_in[w]
    return v_out

def train_BytePairEncoding(
    text: str,
    num_merges: int = 1000,
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
):
    text = normalize_text(text, use_only_lowercase, use_only_alphabetic, make_the_puncs_as_a_single_word)
    vocab = prepare_text_for_bpe(text)

    final_token_vocab = set()
    for w in vocab:
        final_token_vocab.update(w.split())

    merges: List[Tuple[str,str]] = []
    for _ in range(num_merges):
        stats = get_stats(vocab)
        if not stats:
            break
        best_pair = max(stats, key=stats.get)
        final_token_vocab.add("".join(best_pair))
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

    tokens = sorted(list(final_token_vocab))
    token_to_id = {tok: i for i, tok in enumerate(tokens)}
    return merges, tokens, token_to_id

def apply_BytePairEncoding(
    text: str,
    merges: List[Tuple[str,str]],
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
) -> List[str]:
    merge_ranks = {pair: i for i, pair in enumerate(merges)}
    text = normalize_text(text, use_only_lowercase, use_only_alphabetic, make_the_puncs_as_a_single_word)
    encoded: List[str] = []
    for word in text.split(" "):
        if word == "":
            continue
        tokens = list(word) + ["_"]
        while True:
            pairs = []
            for i in range(len(tokens)-1):
                pair = (tokens[i], tokens[i+1])
                if pair in merge_ranks:
                    pairs.append((merge_ranks[pair], i, pair))
            if not pairs:
                break
            pairs.sort()
            _, _, best = pairs[0]
            i = 0
            new_toks = []
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    new_toks.append("".join(best)); i += 2
                else:
                    new_toks.append(tokens[i]); i += 1
            tokens = new_toks
        encoded.extend(tokens)
    return encoded

def calculate_BPE_accuracy(encoded_tokens: List[str]) -> float:
    merged = sum(1 for t in encoded_tokens if len(t) > 1)
    total = len(encoded_tokens)
    return (merged / total) if total else 0.0

SPECIAL_TOKENS = ['<pad>','<bos>','<eos>','<unk>']

class BPETokenizer:
    def __init__(self):
        self.merges: List[Tuple[str,str]] = []
        self.vocab: Dict[str,int] = {}
        self.token_to_id: Dict[str,int] = {}
        self.id_to_token: Dict[int,str] = {}

    def train(self, texts: List[str], num_merges: int = 1000):
        merges, tokens, token_to_id = train_BytePairEncoding(" ".join(texts), num_merges=num_merges)
        self.merges = merges
        self.vocab = {t:i for i,t in enumerate(SPECIAL_TOKENS + tokens)}
        self.token_to_id = dict(self.vocab)
        self.id_to_token = {i:t for t,i in self.vocab.items()}

    def encode(self, s: str, add_bos_eos: bool = True) -> List[int]:
        toks = apply_BytePairEncoding(s, self.merges)
        ids = []
        if add_bos_eos: ids.append(self.token_to_id['<bos>'])
        for t in toks:
            ids.append(self.token_to_id.get(t, self.token_to_id['<unk>']))
        if add_bos_eos: ids.append(self.token_to_id['<eos>'])
        return ids

    def decode(self, ids: List[int]) -> str:
        toks = [self.id_to_token.get(i, '<unk>') for i in ids]
        toks = [t for t in toks if t not in SPECIAL_TOKENS]
        s = "".join(toks).replace("_", " ")
        s = re.sub(r"\s{2,}", " ", s)
        return s.strip()
'''

# -------- utils_ngram.py (Task 2) — no external deps --------
utils_ngram_src = r'''
import math
from typing import List, Tuple, Dict
from collections import Counter

def create_ngrams(tokens: List[int], n: int) -> Counter:
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def build_all_frequencies(train_tokens: List[int], max_n: int) -> Dict[int, Counter]:
    all_freqs: Dict[int, Counter] = {}
    for n in range(1, max_n+1):
        all_freqs[n] = create_ngrams(train_tokens, n)
    return all_freqs

def calculate_mle_prob_smoothed(ngram_tuple: Tuple[int,...], all_frequencies: Dict[int,Counter],
                                n: int, vocab_size: int, alpha: float = 0.01) -> float:
    if n == 1:
        total = sum(all_frequencies[1].values())
        return (all_frequencies[1][ngram_tuple[0]] + alpha) / (total + alpha * vocab_size)
    context = ngram_tuple[:-1]
    context_count = all_frequencies[n-1][context]
    ngram_count   = all_frequencies[n][ngram_tuple]
    return (ngram_count + alpha) / (context_count + alpha * vocab_size)

def get_word_probability(history: List[int], target_word: int,
                         all_frequencies: Dict[int,Counter], lambdas: Dict[int,float],
                         max_n: int, vocab_size: int, alpha: float = 0.01) -> float:
    prob = 0.0
    for n in range(1, max_n+1):
        ctx_len = n - 1
        if len(history) >= ctx_len:
            context = tuple(history[-ctx_len:]) if ctx_len > 0 else tuple()
            ng = context + (target_word,)
            p = calculate_mle_prob_smoothed(ng, all_frequencies, n, vocab_size, alpha)
            prob += lambdas.get(n, 0.0) * p
    return max(prob, 1e-12)

def calculate_perplexity(validation_tokens: List[int], all_frequencies: Dict[int,Counter],
                         lambdas: Dict[int,float], max_n: int, vocab_size: int,
                         alpha: float = 0.01) -> float:
    N = len(validation_tokens)
    if N <= max_n: return float('inf')
    log_sum = 0.0
    count = 0
    for i in range(max_n-1, N-1):
        history = validation_tokens[i-(max_n-1):i]
        target  = validation_tokens[i]
        p = get_word_probability(history, target, all_frequencies, lambdas, max_n, vocab_size, alpha)
        log_sum += math.log(p)
        count += 1
    return math.exp(-log_sum / max(1,count))

def ngram_perplexity(train_tokens: List[int], valid_tokens: List[int], n: int,
                     alpha: float = 0.01) -> float:
    max_n = n
    all_freqs = build_all_frequencies(train_tokens, max_n)
    lambdas = {i: 0.0 for i in range(1, max_n+1)}; lambdas[n] = 1.0
    vocab_size = len(set(train_tokens))
    return calculate_perplexity(valid_tokens, all_freqs, lambdas, max_n, vocab_size, alpha)
'''

# -------- utils_neural_ngram.py (Task 3) --------
utils_neural_src = r'''
import math, torch, torch.nn as nn
from torch.utils.data import DataLoader

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B,T,V)
        loss = None
        if targets is not None:
            B,T,V = logits.shape
            loss = nn.functional.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    def loss(self, x, y):
        _, loss = self.forward(x, y)
        return loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

@torch.no_grad()
def neural_ngram_perplexity(model: nn.Module, loader: DataLoader, device="cpu") -> float:
    model.eval()
    tot_loss, tot_tok = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = model.loss(xb, yb)
        tot_loss += float(loss.item()) * xb.numel()
        tot_tok  += int(xb.numel())
    return math.exp(tot_loss / max(1, tot_tok))
'''

# ---- Write files
Path("utils_bpe.py").write_text(utils_bpe_src, encoding="utf-8")
Path("utils_ngram.py").write_text(utils_ngram_src, encoding="utf-8")
Path("utils_neural_ngram.py").write_text(utils_neural_src, encoding="utf-8")

# ---- Import modules
import utils_bpe, utils_ngram, utils_neural_ngram
importlib.reload(utils_bpe); importlib.reload(utils_ngram); importlib.reload(utils_neural_ngram)

print("✅ Created & imported: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py")

'''

'\n\n# ---- Write files\nPath("utils_bpe.py").write_text(utils_bpe_src, encoding="utf-8")\nPath("utils_ngram.py").write_text(utils_ngram_src, encoding="utf-8")\nPath("utils_neural_ngram.py").write_text(utils_neural_src, encoding="utf-8")\n\n# ---- Import modules\nimport utils_bpe, utils_ngram, utils_neural_ngram\nimportlib.reload(utils_bpe); importlib.reload(utils_ngram); importlib.reload(utils_neural_ngram)\n\nprint("✅ Created & imported: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py")\n\n'

In [5]:
# === Cell 4 — Reuse Task 1–3 as in-notebook modules (fixed & compatible) ===
# This cell writes: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py
# and imports them so later cells just work (with compatibility shims).

from pathlib import Path
import importlib

# -------- utils_bpe.py (Task 1) --------
utils_bpe_src = r'''
import re
from collections import defaultdict
from typing import List, Tuple, Dict

# ---- Task 1 helpers ----

# Normalize casing, optional alphabetic filtering, punctuation spacing, and repeated whitespace.
def normalize_text(
    text: str,
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
) -> str:
    if use_only_lowercase:
        text = text.lower()
    if use_only_alphabetic:
        text = re.sub(r"[^a-z\s]", "", text)
    if make_the_puncs_as_a_single_word:
        text = re.sub(r'([.,!?;:()\[\]{}"\'\-])', r" \1 ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text

# Convert words into character sequences terminated by '_' and count their frequencies.
def prepare_text_for_bpe(text: str, *_args, **_kwargs) -> Dict[str, int]:
    words = text.split(" ")
    vocab = defaultdict(int)
    for w in words:
        if w:
            vocab[" ".join(list(w)) + " _"] += 1
    return vocab

# Count adjacent symbol pairs, weighted by each vocabulary entry's frequency.
def get_stats(vocab: Dict[str, int]):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

# Replace the selected adjacent pair with one merged symbol in every vocabulary entry.
def merge_vocab(pair: Tuple[str,str], v_in: Dict[str,int]) -> Dict[str,int]:
    v_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for w in v_in:
        v_out[p.sub("".join(pair), w)] = v_in[w]
    return v_out

# Learn BPE merges greedily by repeatedly selecting the most frequent adjacent pair.
def train_BytePairEncoding(
    text: str,
    num_merges: int = 1000,
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
):
    text = normalize_text(text, use_only_lowercase, use_only_alphabetic, make_the_puncs_as_a_single_word)
    vocab = prepare_text_for_bpe(text)

    final_token_vocab = set()
    for w in vocab:
        final_token_vocab.update(w.split())

    merges: List[Tuple[str,str]] = []
    for _ in range(num_merges):
        stats = get_stats(vocab)
        if not stats:
            break
        best_pair = max(stats, key=stats.get)
        final_token_vocab.add("".join(best_pair))
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

    tokens = sorted(list(final_token_vocab))
    token_to_id = {tok: i for i, tok in enumerate(tokens)}
    return merges, tokens, token_to_id

# Apply learned merges to new text in the same priority order used during training.
def apply_BytePairEncoding(
    text: str,
    merges: List[Tuple[str,str]],
    use_only_lowercase: bool = True,
    use_only_alphabetic: bool = False,
    make_the_puncs_as_a_single_word: bool = True,
) -> List[str]:
    merge_ranks = {pair: i for i, pair in enumerate(merges)}
    text = normalize_text(text, use_only_lowercase, use_only_alphabetic, make_the_puncs_as_a_single_word)
    encoded: List[str] = []
    for word in text.split(" "):
        if word == "":
            continue
        tokens = list(word) + ["_"]
        while True:
            pairs = []
            for i in range(len(tokens)-1):
                pair = (tokens[i], tokens[i+1])
                if pair in merge_ranks:
                    pairs.append((merge_ranks[pair], i, pair))
            if not pairs:
                break
            pairs.sort()
            _, _, best = pairs[0]
            i = 0
            new_toks = []
            while i < len(tokens):
                if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                    new_toks.append("".join(best)); i += 2
                else:
                    new_toks.append(tokens[i]); i += 1
            tokens = new_toks
        encoded.extend(tokens)
    return encoded

def calculate_BPE_accuracy(encoded_tokens: List[str]) -> float:
    merged = sum(1 for t in encoded_tokens if len(t) > 1)
    total = len(encoded_tokens)
    return (merged / total) if total else 0.0

SPECIAL_TOKENS = ['<pad>','<bos>','<eos>','<unk>']

# Wrap the Task-1 BPE functions in a tokenizer interface used by the later models.
class BPETokenizer:
    def __init__(self):
        self.merges: List[Tuple[str,str]] = []
        self.vocab: Dict[str,int] = {}
        self.token_to_id: Dict[str,int] = {}
        self.id_to_token: Dict[int,str] = {}

    def train(self, texts: List[str], num_merges: int = 1000):
        merges, tokens, token_to_id = train_BytePairEncoding(" ".join(texts), num_merges=num_merges)
        self.merges = merges
        self.vocab = {t:i for i,t in enumerate(SPECIAL_TOKENS + tokens)}
        self.token_to_id = dict(self.vocab)
        self.id_to_token = {i:t for t,i in self.vocab.items()}

    def encode(self, s: str, add_bos_eos: bool = True) -> List[int]:
        toks = apply_BytePairEncoding(s, self.merges)
        ids = []
        if add_bos_eos: ids.append(self.token_to_id['<bos>'])
        for t in toks:
            ids.append(self.token_to_id.get(t, self.token_to_id['<unk>']))
        if add_bos_eos: ids.append(self.token_to_id['<eos>'])
        return ids

    def decode(self, ids: List[int]) -> str:
        toks = [self.id_to_token.get(i, '<unk>') for i in ids]
        toks = [t for t in toks if t not in SPECIAL_TOKENS]
        s = "".join(toks).replace("_", " ")
        s = re.sub(r"\s{2,}", " ", s)
        return s.strip()
'''

# -------- utils_ngram.py (Task 2) — includes compat shim --------
utils_ngram_src = r'''
import math
from typing import List, Tuple, Dict
from collections import Counter

# Count all contiguous n-token sequences in a tokenized corpus.
def create_ngrams(tokens: List[int], n: int) -> Counter:
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

# Precompute frequency tables for every order from unigram through max_n.
def build_all_frequencies(train_tokens: List[int], max_n: int) -> Dict[int, Counter]:
    all_freqs: Dict[int, Counter] = {}
    for n in range(1, max_n+1):
        all_freqs[n] = create_ngrams(train_tokens, n)
    return all_freqs

# Estimate an n-gram probability with additive smoothing to avoid zero-probability events.
def calculate_mle_prob_smoothed(ngram_tuple: Tuple[int,...], all_frequencies: Dict[int,Counter],
                                n: int, vocab_size: int, alpha: float = 0.01) -> float:
    if n == 1:
        total = sum(all_frequencies[1].values())
        return (all_frequencies[1][ngram_tuple[0]] + alpha) / (total + alpha * vocab_size)
    context = ngram_tuple[:-1]
    context_count = all_frequencies[n-1][context]
    ngram_count   = all_frequencies[n][ngram_tuple]
    return (ngram_count + alpha) / (context_count + alpha * vocab_size)

def get_word_probability(history: List[int], target_word: int,
                         all_frequencies: Dict[int,Counter], lambdas: Dict[int,float],
                         max_n: int, vocab_size: int, alpha: float = 0.01) -> float:
    prob = 0.0
    for n in range(1, max_n+1):
        ctx_len = n - 1
        if len(history) >= ctx_len:
            context = tuple(history[-ctx_len:]) if ctx_len > 0 else tuple()
            ng = context + (target_word,)
            p = calculate_mle_prob_smoothed(ng, all_frequencies, n, vocab_size, alpha)
            prob += lambdas.get(n, 0.0) * p
    return max(prob, 1e-12)

# Accumulate validation log-probabilities and convert the average negative log-likelihood to perplexity.
def calculate_perplexity(validation_tokens: List[int], all_frequencies: Dict[int,Counter],
                         lambdas: Dict[int,float], max_n: int, vocab_size: int,
                         alpha: float = 0.01) -> float:
    N = len(validation_tokens)
    if N <= max_n: return float('inf')
    log_sum = 0.0
    count = 0
    for i in range(max_n-1, N):
        history = validation_tokens[i-(max_n-1):i]
        target  = validation_tokens[i]
        p = get_word_probability(history, target, all_frequencies, lambdas, max_n, vocab_size, alpha)
        log_sum += math.log(p)
        count += 1
    return math.exp(-log_sum / max(1,count))

# Build the requested n-gram model on training tokens and score the validation tokens.
def _ngram_perplexity_train_valid(train_tokens: List[int], valid_tokens: List[int], n: int,
                                  alpha: float = 0.01) -> float:
    max_n = n
    all_freqs = build_all_frequencies(train_tokens, max_n)
    lambdas = {i: 0.0 for i in range(1, max_n+1)}; lambdas[n] = 1.0
    vocab_size = len(set(train_tokens))
    return calculate_perplexity(valid_tokens, all_freqs, lambdas, max_n, vocab_size, alpha)

# ---- Compatibility shim ----
def ngram_perplexity(train_tokens, valid_tokens=None, n: int = 1, alpha: float = 0.01, vocab_size=None):
    """
    Preferred: ngram_perplexity(train_tokens, valid_tokens, n, alpha).
    Back-compat: if valid_tokens is None, train==valid (not ideal, but prevents crashes).
    """
    if valid_tokens is None:
        valid_tokens = train_tokens
        if vocab_size is None:
            vocab_size = len(set(train_tokens))
    return _ngram_perplexity_train_valid(train_tokens, valid_tokens, n, alpha)
'''

# -------- utils_neural_ngram.py (Task 3) — includes both classes --------
utils_neural_src = r'''
import math, torch, torch.nn as nn
from torch.utils.data import DataLoader

# Neural baseline: each input token directly indexes logits for the next token.
class BigramLanguageModel(nn.Module):
    """Simple neural bigram: token embedding directly to vocab-sized logits."""
    def __init__(self, vocab_size: int):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B,T,V)
        loss = None
        if targets is not None:
            B,T,V = logits.shape
            loss = nn.functional.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    def loss(self, x, y):
        _, loss = self.forward(x, y)
        return loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Slightly richer baseline: map token embeddings through a linear vocabulary projection.
class NeuralNgramLM(nn.Module):
    """Neural 'ngram' baseline with configurable embedding size -> linear to vocab."""
    def __init__(self, vocab_size: int, n_embd: int = 128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, n_embd)
        self.lin = nn.Linear(n_embd, vocab_size)

    def forward(self, x, targets=None):
        e = self.emb(x)         # (B,T,E)
        logits = self.lin(e)    # (B,T,V)
        loss = None
        if targets is not None:
            B,T,V = logits.shape
            loss = nn.functional.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

    def loss(self, x, y):
        _, loss = self.forward(x, y)
        return loss

# Evaluate average token loss without gradients, then exponentiate it to obtain perplexity.
@torch.no_grad()
def neural_ngram_perplexity(model: nn.Module, loader: DataLoader, device="cpu") -> float:
    model.eval()
    tot_loss, tot_tok = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = model.loss(xb, yb)
        tot_loss += float(loss.item()) * xb.numel()
        tot_tok  += int(xb.numel())
    return math.exp(tot_loss / max(1, tot_tok))
'''

# ---- Write files
Path("utils_bpe.py").write_text(utils_bpe_src, encoding="utf-8")
Path("utils_ngram.py").write_text(utils_ngram_src, encoding="utf-8")
Path("utils_neural_ngram.py").write_text(utils_neural_src, encoding="utf-8")

# ---- Import modules
import utils_bpe, utils_ngram, utils_neural_ngram
importlib.reload(utils_bpe); importlib.reload(utils_ngram); importlib.reload(utils_neural_ngram)

print("✅ Created & imported: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py")
print("   - utils_ngram.ngram_perplexity supports BOTH (train, valid, n, alpha) and (tokens, n, vocab_size) styles.")
print("   - utils_neural_ngram has BOTH BigramLanguageModel() and NeuralNgramLM(n_embd=...).")


✅ Created & imported: utils_bpe.py, utils_ngram.py, utils_neural_ngram.py
   - utils_ngram.ngram_perplexity supports BOTH (train, valid, n, alpha) and (tokens, n, vocab_size) styles.
   - utils_neural_ngram has BOTH BigramLanguageModel() and NeuralNgramLM(n_embd=...).


In [6]:
# Tokenize with your Task-1 BPE and build DataLoaders

# Choose your BPE merge count (use what worked best for you in Task 2)
BPE_MERGES = 1600  # <- adjust (try your top-3 later)

# Train tokenizer on TRAIN text (you already loaded train_text/valid_text/test_text earlier)
# Fit BPE only on the training split so validation and test data do not leak into the vocabulary.
tokenizer = utils_bpe.BPETokenizer()
tokenizer.train([train_text], num_merges=BPE_MERGES)

V = len(tokenizer.vocab)
print("Vocab size:", V)

def encode_corpus(s: str, add_bos_eos=True):
    return tokenizer.encode(s, add_bos_eos=add_bos_eos)

# Encode every split with the same learned tokenizer and include BOS/EOS boundary tokens.
train_ids = encode_corpus(train_text, add_bos_eos=True)
valid_ids = encode_corpus(valid_text, add_bos_eos=True)
test_ids  = encode_corpus(test_text,  add_bos_eos=True)
print("Encoded lengths:", len(train_ids), len(valid_ids), len(test_ids))

# Teacher-forcing dataset
import torch
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

# Keep sequence length and batch size together so all loaders use the same settings.
@dataclass
class LMConfig:
    block_size: int = 128
    batch_size: int = 64

cfg_lm = LMConfig(block_size=128, batch_size=64)

# Create teacher-forcing examples where each target sequence is the input shifted by one token.
class LMDataset(Dataset):
    def __init__(self, ids, block):
        self.ids = ids; self.block = block
    def __len__(self):
        return max(0, len(self.ids) - self.block - 1)
    def __getitem__(self, i):
        x = torch.tensor(self.ids[i:i+self.block], dtype=torch.long)
        y = torch.tensor(self.ids[i+1:i+self.block+1], dtype=torch.long)
        return x, y

# Shuffle training windows, while keeping validation and test loaders deterministic.
train_loader = DataLoader(LMDataset(train_ids, cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(LMDataset(valid_ids, cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=False, drop_last=True)
test_loader  = DataLoader(LMDataset(test_ids,  cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=False, drop_last=True)


Vocab size: 1645
Encoded lengths: 258697 31642 31032


In [7]:
# === Sweep BPE merges using the n-gram baseline; pick top-3 ===
from copy import deepcopy

# Compare several tokenizer sizes while keeping the n-gram order and smoothing fixed.
MERGE_CANDIDATES = [800, 1200, 1600, 2000, 2400]  # tweak as you wish
NGRAM_N = 4
ALPHA = 0.01

# Train a fresh tokenizer for each merge count and measure validation perplexity.
results = []
for m in MERGE_CANDIDATES:
    tok = utils_bpe.BPETokenizer()
    tok.train([train_text], num_merges=m)
    tr_ids = tok.encode(train_text, add_bos_eos=True)
    va_ids = tok.encode(valid_text, add_bos_eos=True)
    ppl = utils_ngram.ngram_perplexity(tr_ids, va_ids, n=NGRAM_N, alpha=ALPHA)
    results.append((m, ppl))
    print(f"BPE merges={m:4d} → n-gram({NGRAM_N}) valid PPL = {ppl:.2f}")

# Sort and pick top-3 with lowest PPL
# Lower perplexity is better, so sort ascending and retain the three best merge counts.
results.sort(key=lambda x: x[1])
top3_merges = [m for m,_ in results[:3]]
print("Top-3 BPE merges by n-gram PPL:", top3_merges)

# (Optional) save quick CSV in the current RUN
# Persist the full sweep so the tokenizer choice can be traced back to its validation result.
if RUN.run_dir is None:
    RUN.new("bpe_sweep_ngram", {"candidates": MERGE_CANDIDATES, "ngram_n": NGRAM_N, "alpha": ALPHA})
for m,ppl in results:
    RUN.log_row("bpe_ngram_valid.csv", {"merges": m, "ngram_n": NGRAM_N, "alpha": ALPHA, "valid_ppl": ppl},
                header=["merges","ngram_n","alpha","valid_ppl"])


BPE merges= 800 → n-gram(4) valid PPL = 179.84
BPE merges=1200 → n-gram(4) valid PPL = 296.11
BPE merges=1600 → n-gram(4) valid PPL = 420.85
BPE merges=2000 → n-gram(4) valid PPL = 543.60
BPE merges=2400 → n-gram(4) valid PPL = 668.54
Top-3 BPE merges by n-gram PPL: [800, 1200, 1600]
Run dir: /content/runs/20260923-235411-bpe_sweep_ngram


# n-gram baseline (Task 2)

In [8]:
# Build on TRAIN, evaluate on VALID
# Use additive smoothing (alpha=0.01) when scoring the 4-gram baseline.
ppl_ngram4 = utils_ngram.ngram_perplexity(train_tokens=train_ids, valid_tokens=valid_ids, n=4, alpha=0.01)
print("Baseline 4-gram PPL(valid):", round(ppl_ngram4, 2))


Baseline 4-gram PPL(valid): 420.85


# Neural baseline (Task 3 style)

In [9]:
# Simple neural bigram baseline (few epochs just to get a number)
import time, math, torch, torch.nn as nn

# Initialize the neural bigram baseline on the selected compute device.
neural_model = utils_neural_ngram.BigramLanguageModel(V).to(DEVICE)
opt = torch.optim.AdamW(neural_model.parameters(), lr=1e-3)

EPOCHS = 2  # increase for a stronger baseline later
# Train for a small fixed number of epochs to obtain a lightweight comparison baseline.
for ep in range(1, EPOCHS+1):
    neural_model.train()
    t0, tot = time.time(), 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        loss = neural_model.loss(xb, yb)
        # Backpropagate the language-model loss, clip gradients, and update the parameters.
        loss.backward()
        nn.utils.clip_grad_norm_(neural_model.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f"neural ep{ep}: train_loss={tot/len(train_loader):.3f} time={time.time()-t0:.1f}s")

# Evaluate validation perplexity after training using the same token-level objective.
ppl_neural = utils_neural_ngram.neural_ngram_perplexity(neural_model, valid_loader, device=DEVICE)
print("Neural baseline PPL(valid):", round(ppl_neural, 2))


neural ep1: train_loss=5.999 time=26.5s
neural ep2: train_loss=4.328 time=25.5s
Neural baseline PPL(valid): 125.04


In [10]:
# Define GPT only if not already defined above
import math, torch.nn.functional as F
from dataclasses import dataclass as _dataclass

if "GPT" not in globals():
    # Store all architecture hyperparameters needed to construct the GPT model.
    @_dataclass
    class GPTConfig:
        vocab_size: int
        n_embd: int = 384
        n_head: int = 6
        n_layer: int = 6
        block_size: int = 128
        dropout: float = 0.1
        tie_weights: bool = True

    # Multi-head causal self-attention: each position can attend only to itself and earlier positions.
    class CausalSelfAttention(nn.Module):
        def __init__(self, cfg: GPTConfig):
            super().__init__()
            assert cfg.n_embd % cfg.n_head == 0
            self.cfg = cfg
            self.n_head = cfg.n_head
            self.head_dim = cfg.n_embd // cfg.n_head
            self.qkv = nn.Linear(cfg.n_embd, 3*cfg.n_embd)
            self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
            self.attn_drop = nn.Dropout(cfg.dropout)
            self.resid_drop = nn.Dropout(cfg.dropout)
            self.register_buffer("mask", torch.tril(torch.ones(cfg.block_size, cfg.block_size)).view(1,1,cfg.block_size,cfg.block_size))
        def forward(self, x):
            B,T,C = x.shape
            # Project once, then split the result into query, key, and value tensors.
            qkv = self.qkv(x)
            q,k,v = qkv.split(C, dim=2)
            q = q.view(B,T,self.n_head,self.head_dim).transpose(1,2)
            k = k.view(B,T,self.n_head,self.head_dim).transpose(1,2)
            v = v.view(B,T,self.n_head,self.head_dim).transpose(1,2)
            # Compute scaled attention scores and mask future positions before softmax.
            att = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
            att = att.masked_fill(self.mask[:,:,:T,:T]==0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_drop(att)
            y = att @ v
            y = y.transpose(1,2).contiguous().view(B,T,C)
            y = self.resid_drop(self.proj(y))
            return y

    # Position-wise feed-forward network used inside each transformer block.
    class MLP(nn.Module):
        def __init__(self, cfg: GPTConfig):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(cfg.n_embd, 4*cfg.n_embd),
                nn.GELU(),
                nn.Linear(4*cfg.n_embd, cfg.n_embd),
                nn.Dropout(cfg.dropout),
            )
        def forward(self, x): return self.net(x)

    # Pre-normalized transformer block with residual attention and MLP connections.
    class Block(nn.Module):
        def __init__(self, cfg: GPTConfig):
            super().__init__()
            self.ln1 = nn.LayerNorm(cfg.n_embd)
            self.attn = CausalSelfAttention(cfg)
            self.ln2 = nn.LayerNorm(cfg.n_embd)
            self.mlp = MLP(cfg)
        def forward(self, x):
            x = x + self.attn(self.ln1(x))
            x = x + self.mlp(self.ln2(x))
            return x

    # Decoder-only transformer language model with learned token and positional embeddings.
    class GPT(nn.Module):
        def __init__(self, cfg: GPTConfig):
            super().__init__()
            self.cfg = cfg
            # Represent token identity and sequence position in the same embedding space.
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
            self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
            self.drop = nn.Dropout(cfg.dropout)
            self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
            self.ln_f = nn.LayerNorm(cfg.n_embd)
            self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
            # Optionally share input-token embedding weights with the output vocabulary projection.
            if cfg.tie_weights:
                self.head.weight = self.tok_emb.weight
            self.apply(self._init)
        # Initialize trainable linear and embedding weights with the standard small normal distribution.
        def _init(self, m):
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        # Produce next-token logits and, when targets are supplied, cross-entropy training loss.
        def forward(self, idx, targets=None):
            B,T = idx.shape
            assert T <= self.cfg.block_size
            pos = torch.arange(0,T,device=idx.device).unsqueeze(0)
            x = self.tok_emb(idx) + self.pos_emb(pos)
            x = self.drop(x)
            for blk in self.blocks: x = blk(x)
            x = self.ln_f(x)
            logits = self.head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss

    # Average validation/test loss without gradient tracking.
    @torch.no_grad()
    def evaluate(model: GPT, loader: DataLoader, device="cuda"):
        model.eval()
        losses=[]
        for xb,yb in loader:
            xb,yb = xb.to(device), yb.to(device)
            _, loss = model(xb,yb); losses.append(loss.item())
        model.train()
        return float(np.mean(losses))

    # Autoregressively sample one token at a time with temperature scaling and optional top-k filtering.
    @torch.no_grad()
    def generate(model: GPT, tokenizer, prompt: str, max_new_tokens=200, temperature=1.0, top_k: int=None, device="cuda"):
        model.eval()
        idx = torch.tensor([tokenizer.encode(prompt, add_bos_eos=False)], dtype=torch.long, device=device)
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -model.cfg.block_size:]
            logits, _ = model(idx_cond)
            logits = logits[:,-1,:] / max(1e-8, temperature)
            if top_k is not None:
                v,_ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:,[-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == tokenizer.token_to_id.get('<eos>', -1):
                break
        return tokenizer.decode(idx[0].tolist())

print("GPT classes ready.")


GPT classes ready.


In [11]:
# === Train GPT for each of the top-3 merge counts, pick the best on VALID ===
import time, math, json, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt

# Collect one summary tuple per tokenizer setting for the final validation-based comparison.
gpt_runs = []  # (merges, run_dir, best_val, test_ppl)

# Retrain the tokenizer and GPT from scratch for each of the three selected merge counts.
for merges in top3_merges:
    # -- (1) Tokenizer & data
    tokenizer = utils_bpe.BPETokenizer()
    tokenizer.train([train_text], num_merges=merges)

    V = len(tokenizer.vocab)
    def enc(s): return tokenizer.encode(s, add_bos_eos=True)
    train_ids = enc(train_text); valid_ids = enc(valid_text); test_ids = enc(test_text)

    from torch.utils.data import DataLoader, Dataset
    class LMDataset(Dataset):
        def __init__(self, ids, block): self.ids=ids; self.block=block
        def __len__(self): return len(self.ids)-self.block-1
        def __getitem__(self, i):
            x = torch.tensor(self.ids[i:i+cfg_lm.block_size], dtype=torch.long)
            y = torch.tensor(self.ids[i+1:i+cfg_lm.block_size+1], dtype=torch.long)
            return x,y

    train_loader = DataLoader(LMDataset(train_ids, cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=True, drop_last=True)
    valid_loader = DataLoader(LMDataset(valid_ids, cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=False, drop_last=True)
    test_loader  = DataLoader(LMDataset(test_ids,  cfg_lm.block_size), batch_size=cfg_lm.batch_size, shuffle=False, drop_last=True)

    # -- (2) Config for this run (keep budget modest but same across merges)
    # Use the same model/training budget across merge counts for a fair comparison.
    cfg = {
      "seed": 1337,
      "data": {"bpe_merges": merges, "block_size": cfg_lm.block_size},
      "model": {"n_embd": 256, "n_head": 4, "n_layer": 4, "dropout": 0.1},
      "train": {"batch_size": cfg_lm.batch_size, "lr": 3e-4, "max_steps": 1000, "eval_every": 100, "patience": 4,
                "weight_decay": 0.01, "grad_clip": 1.0},
      "gen": {"temperature": 0.9, "top_k": 50, "max_new_tokens": 160}
    }
    run_dir = RUN.new(f"gpt_m{merges}", cfg)

    gcfg = GPTConfig(
        vocab_size=V, n_embd=cfg["model"]["n_embd"], n_head=cfg["model"]["n_head"], n_layer=cfg["model"]["n_layer"],
        block_size=cfg["data"]["block_size"], dropout=cfg["model"]["dropout"], tie_weights=True
    )
    # Build a fresh GPT and optimizer for this tokenizer configuration.
    model = GPT(gcfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["train"]["lr"], betas=(0.9,0.95), weight_decay=cfg["train"]["weight_decay"])

    best_val = float('inf'); bad=0; steps=0; val_hist=[]

    # Optimize until the step budget is exhausted or validation early stopping triggers.
    while steps < cfg["train"]["max_steps"]:
        for xb,yb in train_loader:
            steps += 1
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            _, loss = model(xb,yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["train"]["grad_clip"])
            opt.step()

            # Periodically evaluate validation loss and checkpoint improvements.
            if steps % cfg["train"]["eval_every"] == 0:
                val_loss = evaluate(model, valid_loader, device=DEVICE)
                val_ppl  = math.exp(val_loss)
                val_hist.append((steps, val_loss))
                RUN.log_row("valid.csv", {"steps":steps, "val_loss":val_loss, "val_ppl":val_ppl}, header=["steps","val_loss","val_ppl"])
                if val_loss < best_val - 1e-6:
                    best_val = val_loss; bad=0
                    RUN.save_ckpt({"model":model.state_dict(),"opt":opt.state_dict(),"steps":steps,"val_loss":val_loss}, steps, tag="gpt")
                else:
                    bad += 1
                    if bad >= cfg["train"]["patience"]:
                        print(f"[merges={merges}] Early stopping at step {steps}. Best val={best_val:.3f}")
                        steps = cfg["train"]["max_steps"]
                        break
            if steps >= cfg["train"]["max_steps"]:
                break

    # Plot validation curve
    if val_hist:
        xs, vs = zip(*val_hist)
        fig = plt.figure()
        plt.plot(xs, vs); plt.xlabel("step"); plt.ylabel("val loss"); plt.title(f"GPT val loss (merges={merges})")
        RUN.save_plot(fig, f"val_loss_m{merges}.png")

    # Load best (already in RAM), compute TEST ppl once
# Score the held-out test split after finishing this merge-count experiment.
    test_loss = evaluate(model, test_loader, device=DEVICE)
    test_ppl  = math.exp(test_loss)
    RUN.log_row("summary.csv", {"merges":merges, "best_val_loss":best_val, "test_loss":test_loss, "test_ppl":test_ppl},
                header=["merges","best_val_loss","test_loss","test_ppl"])
    print(f"[merges={merges}] TEST ppl = {test_ppl:.2f}")
    gpt_runs.append((merges, run_dir, best_val, test_ppl))

# Sort by best validation loss and print a summary table
gpt_runs.sort(key=lambda x: x[2])
print("\n=== GPT runs (sorted by best VAL loss) ===")
for m,rd,bv,tp in gpt_runs:
    print(f"merges={m:4d} | best_val_loss={bv:.3f} | test_ppl={tp:.2f} | run={rd}")
best_merges = gpt_runs[0][0]
print("\nBest merges for GPT:", best_merges)


Run dir: /content/runs/20260923-235533-gpt_m800
[merges=800] TEST ppl = 48.82
Run dir: /content/runs/20260923-235952-gpt_m1200
[merges=1200] TEST ppl = 64.21
Run dir: /content/runs/20260924-000412-gpt_m1600
[merges=1600] TEST ppl = 75.93

=== GPT runs (sorted by best VAL loss) ===
merges= 800 | best_val_loss=3.900 | test_ppl=48.82 | run=/content/runs/20260923-235533-gpt_m800
merges=1200 | best_val_loss=4.127 | test_ppl=64.21 | run=/content/runs/20260923-235952-gpt_m1200
merges=1600 | best_val_loss=4.282 | test_ppl=75.93 | run=/content/runs/20260924-000412-gpt_m1600

Best merges for GPT: 800


# Train GPT, save to ./runs/, plot, evaluate, generate

In [12]:
# Train GPT (small demo config; bump for stronger results)
cfg = {
    "seed": 1337,
    "data": {"bpe_merges": BPE_MERGES, "block_size": cfg_lm.block_size},
    "model": {"n_embd": 256, "n_head": 4, "n_layer": 4, "dropout": 0.1},
    "train": {"lr": 3e-4, "max_steps": 1200, "eval_every": 150, "patience": 5, "batch_size": cfg_lm.batch_size},
    "gen": {"temperature": 0.9, "top_k": 50, "max_new_tokens": 160}
}
# Create a dedicated run directory so this training run's artifacts stay grouped together.
RUN_DIR = RUN.new("gpt_task4", cfg)

gcfg = GPTConfig(
    vocab_size=V,
    n_embd=cfg["model"]["n_embd"],
    n_head=cfg["model"]["n_head"],
    n_layer=cfg["model"]["n_layer"],
    block_size=cfg_lm.block_size,
    dropout=cfg["model"]["dropout"],
    tie_weights=True
)
# Instantiate the configured GPT and optimize it with AdamW.
model = GPT(gcfg).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=cfg["train"]["lr"], betas=(0.9,0.95), weight_decay=0.01)

best_val = float('inf'); bad=0; steps=0
val_hist=[]

# Train in mini-batches while counting optimizer steps rather than full epochs.
while steps < cfg["train"]["max_steps"]:
    for xb,yb in train_loader:
        steps += 1
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        _, loss = model(xb,yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        # Evaluate periodically, save improvements, and stop after repeated non-improvements.
        if steps % cfg["train"]["eval_every"] == 0:
            val_loss = evaluate(model, valid_loader, device=DEVICE)
            val_ppl  = math.exp(val_loss)
            val_hist.append((steps, val_loss))
            RUN.log_row("train.csv", {"steps":steps, "loss":float(loss.item())}, header=["steps","loss"])
            RUN.log_row("valid.csv", {"steps":steps, "val_loss":val_loss, "val_ppl":val_ppl}, header=["steps","val_loss","val_ppl"])
            print(f"step {steps}: train_loss={loss.item():.3f}  val_loss={val_loss:.3f}  val_ppl={val_ppl:.2f}")

            if val_loss < best_val - 1e-6:
                best_val = val_loss; bad=0
                RUN.save_ckpt({"model":model.state_dict(),"opt":opt.state_dict(),"steps":steps,"val_loss":val_loss}, steps, tag="gpt")
            else:
                bad += 1
                if bad >= cfg["train"]["patience"]:
                    print("Early stopping.")
                    steps = cfg["train"]["max_steps"]
                    break
        if steps >= cfg["train"]["max_steps"]:
            break

# Plot validation loss
if val_hist:
    xs, vs = zip(*val_hist)
    fig = plt.figure()
    plt.plot(xs, vs, label="val loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Validation loss (GPT)")
    RUN.save_plot(fig, "val_loss.png")

# Final test PPL
# Report final held-out perplexity after the training loop.
test_loss = evaluate(model, test_loader, device=DEVICE)
test_ppl  = math.exp(test_loss)
print(f"GPT test: loss={test_loss:.3f}  ppl={test_ppl:.2f}")

# Generate a sample
# Generate an autoregressive sample from a short Shakespeare-style prompt.
prompt = "To be, or not to be"
sample = generate(model, tokenizer, prompt=prompt, **cfg["gen"])
print("---- SAMPLE ----")
print(sample)


Run dir: /content/runs/20260924-000749-gpt_task4
step 150: train_loss=5.205  val_loss=5.221  val_ppl=185.06
step 300: train_loss=4.802  val_loss=4.821  val_ppl=124.08
step 450: train_loss=4.520  val_loss=4.635  val_ppl=103.03
step 600: train_loss=4.415  val_loss=4.514  val_ppl=91.26
step 750: train_loss=4.253  val_loss=4.418  val_ppl=82.92
step 900: train_loss=4.072  val_loss=4.341  val_ppl=76.77
step 1050: train_loss=3.931  val_loss=4.265  val_ppl=71.16
step 1200: train_loss=3.908  val_loss=4.220  val_ppl=68.03
GPT test: loss=4.274  ppl=71.78
---- SAMPLE ----
to be , or not to be the conquest ; but , nor the sounded of this way shall have birage for antony . cleopatra i ' ll not the noble antony . iras it is , which you in egypt will not that you have heard , but , octavius caesar ' tis true . cleopatra that i have , so , the gods should have done . charmian no , madam , caesar ay , madam . cleopatra what you do : and we were , the general . charmian there ' s the queen . cleopatra my

# Compare perplexities

In [13]:
# Print the main baseline and GPT perplexities together for quick comparison.
print("Perplexities:")
print(" - n-gram (4-gram, valid):", round(ppl_ngram4, 2))
print(" - neural baseline (valid):", round(ppl_neural, 2))
print(" - GPT (test):", round(test_ppl, 2))


Perplexities:
 - n-gram (4-gram, valid): 420.85
 - neural baseline (valid): 125.04
 - GPT (test): 71.78


## Results and interpretation

The recorded BPE sweep selected **800, 1200, and 1600 merges** for GPT training, based on the 4-gram validation perplexities (179.84, 296.11, and 420.85, respectively). Among these three runs, the notebook selected **800 merges** by its lowest recorded validation loss (3.900); its recorded test perplexity was **48.82**. The 1200- and 1600-merge runs recorded test perplexities of **64.21** and **75.93**.

The separate final GPT run uses **1600 merges** and recorded a validation perplexity of **68.03** at step 1200 and a test perplexity of **71.78**. For the same 1600-merge tokenizer, the 4-gram baseline recorded **420.85 validation perplexity** and the neural bigram baseline **125.04 validation perplexity**. These values suggest the GPT model learned more context, but its number is measured on the **test split**, while the two baseline numbers are measured on **validation**; the final three figures are therefore not a matched-split comparison. Perplexity also depends on tokenization, so figures from different BPE merge counts should be interpreted with that in mind.

The generated sample contains recognizable Shakespeare-style dialogue and punctuation, alongside grammatical errors and repetition. It is a qualitative example, not an additional perplexity measurement.
